# Lesson 02: Core Concepts, Primitives & JSON-RPC 2.0

In this notebook, you will:
1. Register all three MCP Server Primitives (`@mcp.tool()`, `@mcp.resource()`, and `@mcp.prompt()`) using Python `FastMCP`.
2. Inspect Tool Annotations (`readOnlyHint`, `destructiveHint`) and JSON Schema 2020-12 definitions.
3. Test JSON-RPC 2.0 error handling (`-32602`).

In [ ]:
!pip install -q -U "mcp<2" pydantic

In [ ]:
import json
from mcp.server.fastmcp import FastMCP
from mcp.types import ToolAnnotations

mcp = FastMCP("PrimitivesDemo")

CATALOG = {
    "sku-101": {"name": "Trail Running Shoes", "price": 129.0, "stock": 14},
    "sku-102": {"name": "Waterproof Jacket", "price": 189.0, "stock": 6},
}

@mcp.resource("catalog://products/all")
def get_all_products() -> str:
    """Return the full product catalog as JSON."""
    return json.dumps(CATALOG, indent=2)

@mcp.prompt()
def product_pitch(product_name: str, audience: str) -> str:
    """Reusable prompt template for product copy."""
    return f"Write a concise 2-sentence pitch for {product_name} aimed at {audience}."

@mcp.tool(annotations=ToolAnnotations(readOnlyHint=True, idempotentHint=True))
def search_catalog(query: str, max_price: float = 500.0) -> list[dict]:
    """Search products by keyword and maximum price."""
    q = query.lower()
    return [
        {"sku": k, **v}
        for k, v in CATALOG.items()
        if q in v["name"].lower() and v["price"] <= max_price
    ]

print("Resource sample:", get_all_products())
print("Prompt sample:", product_pitch("Trail Running Shoes", "marathoners"))
print("Tool search result:", search_catalog("shoes", 150.0))

### Challenge Exercise
Add a write tool `@mcp.tool(annotations=ToolAnnotations(destructiveHint=True))` named `archive_sku(sku: str)` that removes an item from `CATALOG` or raises a `ValueError` if the SKU does not exist.